# LeNet5 / MNIST: predictions under input shift

Two shift sweeps over the seeded 1k MNIST test subset, one prediction engine,
up to four models compared:

- **MAP** point estimate (`x_ref`) -- the Boomerang reference point.
- **SGD** point estimate -- a genuine frequentist LeNet5 trained with plain
  minibatch SGD, no prior (`sazz.gpu_friendly.scripts.lenet_sgd`, same 60k
  pool / 80-20 split / seed as the MAP). Izmailov et al. (2021) Fig 15 use
  SGD in exactly this baseline role.
- **Sticky Zig-Zag** and **Sticky Boomerang** posteriors.

Point estimates flow through the pipeline as single-draw "posteriors"
(dotted lines); the samplers use `N_DRAWS_POOL` draws (solid).

1. **Rotation** -- rotate each image by `ANGLES` degrees (black fill,
   bilinear; stays in [0, 1]). Ovadia et al. (2019) / Izmailov et al. (2021)
   rotated-MNIST. *Label-changing:* a 180-rotated 6 is a 9.
2. **Per-pixel Gaussian noise** -- add unclipped N(0, sigma^2) with sigma in
   de-normalised [0, 1] pixel units (Izmailov et al. 2021, App. F Fig 15
   convention). *Label-preserving.* `SIGMAS` is chosen so the MAP curve
   spans clean -> ~chance on this LeNet (their Fig 15 uses a width-256
   MLP, less noise-sensitive, so curves are not numerically comparable at
   matched sigma -- the point-estimate rows are the honest in-figure
   baseline instead).

For each digit in `CLASSES` we pool `N_PER_CLASS` test images, apply the
shift at every level, push draws (or the single point vector) through, and
aggregate the posterior-mean P(digit) plus accuracy / P(true) / confidence /
entropy / ECE over the pool.

Figures:
- `shift_figure(kind)` -- one main-text figure per shift (noise, rotation),
  each with per-digit accuracy & P(true) plus pooled calibration and entropy.
- `bar_grid_appendix`  -- full P(digit) bar grid (samplers only), one per
  shift, for the appendix.


## 1. Config

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
from pathlib import Path

import numpy as np
import torch
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt

if Path.cwd().name == "notebooks":
    os.chdir("..")
print("cwd:", Path.cwd())

DEVICE = (torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cuda") if torch.cuda.is_available()
          else torch.device("cpu"))
DTYPE = torch.float32
print("device:", DEVICE)

plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 11, "figure.dpi": 120,
})

RUN_DIR = Path("results/paper/mnist_cnn/split_00")
RUN_SPECS = [("zigzag", "grid_sticky_zigzag.pt"),
             ("boomerang", "grid_sticky_boomerang.pt")]
MAP_REF_PATH = Path("results/maps/lenet_reference_N60000_pruned_refit_N60k.pt")
SGD_REF_PATH = Path("results/maps/lenet_sgd_N60000_epochs40.pt")

CLASSES      = [0, 2, 4, 6, 8]
N_PER_CLASS  = 10_000
N_DRAWS_POOL = 1_000
POOL_SEED    = 0
NOISE_SEED   = 12345

ANGLES = [0, 15, 30, 45, 60, 90, 120, 150, 180]                  # degrees
# sigma in de-normalised [0,1] pixel units, UNCLAMPED (Izmailov et al. 2021,
# App. F convention). Grid chosen so the MAP curve spans clean -> ~chance:
# MAP acc ~ 0.99 / 0.97 / 0.85 / 0.66 / 0.43 / 0.28 / 0.21 at these levels.
SIGMAS = [0.0, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 5.0]

MNIST_MEAN, MNIST_STD = 0.1307, 0.3081
PRIOR_STD_W, PRIOR_STD_B, FAN_IN_SCALING = 2.0, 2.0, True
BASE_SEED = 42

SAVE_DIR = Path("results/plots/MNIST")
# SAVE_DIR.mkdir(parents=True, exist_ok=True)

TRUE_GREEN = "#2CA02C"
SAMPLER_COLORS = {"zigzag": "#4C72B0", "boomerang": "#DD8452",
                  "map": "0.35", "sgd": "0.55"}


## 2. Load posteriors, MAP reference, data, prediction engine

LeNet5 is `tanh` + `avg`-pool (read from the checkpoint), no BatchNorm, so
every parameter is set directly from a posterior draw.

In [ ]:
runs = {}
for label, fname in RUN_SPECS:
    p = RUN_DIR / fname
    if not p.exists():
        print(f"[{label}] missing {p} -- skipped"); continue
    runs[label] = torch.load(p, map_location="cpu", weights_only=False)
    ck = runs[label]
    print(f"[{label}] {ck['samples'].shape[0]} draws  test_acc(ckpt)={ck['test_accuracy']:.4f}  "
          f"pool={ck['pool']}  act={ck['activation']}")
assert runs, f"no run files under {RUN_DIR}"

_ck0 = next(iter(runs.values()))
POOL, ACTIVATION = _ck0["pool"], _ck0["activation"]

map_ck = torch.load(MAP_REF_PATH, map_location="cpu", weights_only=False)
X_REF = map_ck["x_ref"].to(DTYPE)
for label, ck in runs.items():
    dmax = (ck["x_ref"].to(DTYPE) - X_REF).abs().max().item()
    assert dmax < 1e-5, f"[{label}] x_ref mismatch with MAP file (max|delta|={dmax:.2e})"

# Point estimates as one-"draw" runs so they flow through run_sweep and every
# figure. Their "posterior" is a point mass: lo == hi == mean, "confidence"
# is the single softmax's max-prob. MAP is the Boomerang reference point;
# SGD is the genuine frequentist baseline (Izmailov et al. 2021 Fig 15 role).
_order = ["map", "sgd", "zigzag", "boomerang"]

runs["map"] = {"samples": X_REF.unsqueeze(0)}
print(f"MAP ref OK: D={X_REF.shape[0]}  test_acc={map_ck['test_acc']:.4f}  -> runs['map'] (1 draw)")

if SGD_REF_PATH.exists():
    sgd_ck = torch.load(SGD_REF_PATH, map_location="cpu", weights_only=False)
    W_SGD = sgd_ck["x_ref"].to(DTYPE)
    assert W_SGD.shape[0] == X_REF.shape[0], "SGD checkpoint D mismatch"
    assert sgd_ck["activation"] == ACTIVATION and sgd_ck["pool"] == POOL, \
        f"SGD arch mismatch: {sgd_ck['activation']}/{sgd_ck['pool']} vs {ACTIVATION}/{POOL}"
    runs["sgd"] = {"samples": W_SGD.unsqueeze(0)}
    print(f"SGD ref OK: D={W_SGD.shape[0]}  test_acc={sgd_ck['test_acc']:.4f}  "
          f"(epochs={sgd_ck['epochs']}, wd={sgd_ck['weight_decay']})  -> runs['sgd'] (1 draw)")
else:
    print(f"[sgd] missing {SGD_REF_PATH} -- skipped (run sazz.gpu_friendly.scripts.lenet_sgd)")

# fix a consistent model order for every downstream figure
runs = {k: runs[k] for k in _order if k in runs}
print("models:", list(runs))


In [ ]:
from sazz.gpu_friendly.scripts.fast_mnist_cnn import load_mnist_subset

_data = load_mnist_subset(60_000, 10_000, BASE_SEED, Path("datasets"),
                          dtype=DTYPE, device="cpu")
X_test, y_test = _data["X_test"], _data["y_test"]   # [N,1,28,28] normalised, CPU
print("X_test:", tuple(X_test.shape))
print("class counts:", {c: int((y_test == c).sum()) for c in CLASSES})

In [ ]:
from sazz.gpu_friendly.models.neural_networks import LeNet5
from sazz.gpu_friendly.models.model import BayesianModule
from sazz.gpu_friendly.models.priors import build_fan_in_prior_precision

_module = LeNet5(activation=ACTIVATION, pool=POOL).to(dtype=DTYPE, device=DEVICE)
_module.eval()
_prec = build_fan_in_prior_precision(_module, PRIOR_STD_W, PRIOR_STD_B,
                                     FAN_IN_SCALING, dtype=DTYPE, device=DEVICE)
_bm = BayesianModule.build(_module, likelihood="categorical",
                           X=X_test[:2].to(DEVICE), y=y_test[:2].to(DEVICE),
                           prior_precision=_prec, dtype=DTYPE, device=DEVICE)
_pdf = _bm.param_dict_fn


@torch.no_grad()
def predict_probs(beta, X, bs=512):
    beta = beta.to(dtype=DTYPE, device=DEVICE)
    out = []
    for i in range(0, X.shape[0], bs):
        xb = X[i:i + bs].to(dtype=DTYPE, device=DEVICE)
        out.append(torch.softmax(
            torch.func.functional_call(_bm.module, _pdf(beta), (xb,)), -1).cpu())
    return torch.cat(out)


def rotate_batch(X_norm, angle_deg, **_):
    if angle_deg == 0:
        return X_norm.clone()
    img01 = X_norm * MNIST_STD + MNIST_MEAN
    img01 = TF.rotate(img01, float(angle_deg),
                      interpolation=TF.InterpolationMode.BILINEAR, fill=0.0)
    return (img01 - MNIST_MEAN) / MNIST_STD


# def noise_batch(X_norm, sigma, seed=0):
#     if sigma == 0.0:
#         return X_norm.clone()
#     g = torch.Generator().manual_seed(seed)
#     img01 = X_norm * MNIST_STD + MNIST_MEAN
#     img01 = (img01 + sigma * torch.randn(X_norm.shape, generator=g)).clamp(0, 1)
#     return (img01 - MNIST_MEAN) / MNIST_STD
def noise_batch(X_norm, sigma, seed=0):
    """Additive Gaussian noise, sigma in de-normalised [0,1] pixel units,
    NO clamp (network inputs, not display images). Izmailov et al. (2021)
    Fig 15 convention (unclipped N(0, sigma^2 I) on [0,1] MNIST)."""
    if sigma == 0.0:
        return X_norm.clone()
    g = torch.Generator().manual_seed(seed)
    return X_norm + (sigma / MNIST_STD) * torch.randn(X_norm.shape, generator=g)



def denorm(x_norm):
    return (x_norm * MNIST_STD + MNIST_MEAN).clamp(0, 1).squeeze().numpy()

## 3. Sweep pipeline

`run_sweep(kind)` builds the pooled shifted image set, runs `N_DRAWS_POOL`
draws per sampler, and returns `(levels, spans, big_X, agg)` where
`agg[label][(digit, level)]` holds the posterior-mean `mean` / `lo` / `hi`
P(digit) vectors plus `acc`, `p_true`, `n_img`.

In [ ]:
SWEEPS = {
    "rotation": dict(levels=ANGLES, op=rotate_batch, xlabel="Rotation (degrees)"),
    "noise":    dict(levels=SIGMAS, op=noise_batch,  xlabel=r"Pixel-noise $\sigma$"),
}


def _block_metrics(block, c):
    """Per-image-pool metrics for one (model, digit, level) cell.
    `block` is [n_img, 10] posterior-mean probs; `c` is the true digit.

    acc / p_true  -- robustness and confidence-in-correct-class (as before).
    conf          -- mean max-prob = confidence in the PREDICTED class.
                     conf > acc is overconfidence; conf ~ acc is calibrated.
    entropy       -- mean predictive entropy (nats). A Bayesian model
                     should widen (entropy up) under shift; a point
                     estimate that stays sharp while wrong does not.
    ece           -- expected calibration error over this pool (15 equal-
                     width confidence bins), the usual single calibration
                     scalar.
    """
    pred = block.argmax(1)
    conf = block.max(1)
    correct = (pred == c).astype(float)
    ent = -(block * np.log(np.clip(block, 1e-12, 1.0))).sum(1)

    bins = np.linspace(0.0, 1.0, 16)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, 14)
    ece = 0.0
    n = len(block)
    for b in range(15):
        m = idx == b
        if m.any():
            ece += (m.sum() / n) * abs(correct[m].mean() - conf[m].mean())

    return dict(
        mean=block.mean(0),
        lo=np.percentile(block, 10, axis=0),
        hi=np.percentile(block, 90, axis=0),
        acc=float(correct.mean()),
        p_true=float(block[:, c].mean()),
        conf=float(conf.mean()),
        entropy=float(ent.mean()),
        ece=float(ece),
        n_img=len(block),
    )


def run_sweep(kind):
    levels, op = SWEEPS[kind]["levels"], SWEEPS[kind]["op"]

    rng = np.random.default_rng(POOL_SEED)
    pool_idx = {c: rng.choice(np.where(y_test.numpy() == c)[0],
                              size=min(N_PER_CLASS, int((y_test == c).sum())),
                              replace=False)
                for c in CLASSES}

    big_X, spans, row = [], {}, 0
    for c in CLASSES:
        Xc = X_test[pool_idx[c]]
        for li, lv in enumerate(levels):
            big_X.append(op(Xc, lv, seed=NOISE_SEED + 1000 * li + c))
            spans[(c, lv)] = (row, row + len(Xc)); row += len(Xc)
    big_X = torch.cat(big_X)

    agg = {label: {} for label in runs}
    for label, ck in runs.items():
        S = ck["samples"]
        g = torch.Generator().manual_seed(0)
        draw_idx = torch.randperm(S.shape[0], generator=g)[:min(N_DRAWS_POOL, S.shape[0])]
        acc = torch.zeros(big_X.shape[0], 10)
        for j in draw_idx:
            acc += predict_probs(S[j], big_X)
        P = (acc / len(draw_idx)).numpy()
        for c in CLASSES:
            for lv in levels:
                a, b = spans[(c, lv)]
                agg[label][(c, lv)] = _block_metrics(P[a:b], c)
        # pooled over all digits in CLASSES: one calibration curve per model.
        # for a fixed level, stack every digit's block and its true labels.
        for lv in levels:
            blocks, trues = [], []
            for c in CLASSES:
                a, b = spans[(c, lv)]
                blocks.append(P[a:b]); trues.append(np.full(b - a, c))
            Pool = np.concatenate(blocks); tru = np.concatenate(trues)
            pred = Pool.argmax(1); cf = Pool.max(1)
            corr = (pred == tru).astype(float)
            ent = -(Pool * np.log(np.clip(Pool, 1e-12, 1.0))).sum(1)
            bins = np.linspace(0.0, 1.0, 16)
            bi = np.clip(np.digitize(cf, bins) - 1, 0, 14)
            ece = sum((( bi == k).sum() / len(Pool)) *
                      abs(corr[bi == k].mean() - cf[bi == k].mean())
                      for k in range(15) if (bi == k).any())
            agg[label][("pool", lv)] = dict(
                acc=float(corr.mean()), conf=float(cf.mean()),
                p_true=float(Pool[np.arange(len(Pool)), tru].mean()),
                entropy=float(ent.mean()), ece=float(ece), n_img=len(Pool),
            )
    print(f"[{kind}] {len(runs)} models x {len(CLASSES)} digits x {len(levels)} levels")
    return levels, spans, big_X, agg


def show_examples(kind, levels, spans, big_X):
    fig, axes = plt.subplots(len(CLASSES), len(levels),
                             figsize=(2.1 * len(levels), 2.1 * len(CLASSES)), squeeze=False)
    for r, c in enumerate(CLASSES):
        for cc, lv in enumerate(levels):
            a, _ = spans[(c, lv)]
            ax = axes[r][cc]
            ax.imshow(denorm(big_X[a]), cmap="gray", vmin=0, vmax=1)
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(f"{lv:g}" + ("" if kind == "noise" else " deg")
                             + ("  (clean)" if cc == 0 else ""), fontsize=10)
            if cc == 0:
                ax.set_ylabel(f"digit {c}", fontsize=11)
    fig.suptitle(f"{kind}: one example image per digit per level", y=1.01)
    fig.tight_layout(); plt.show()

## 4. Run both sweeps

In [ ]:
rot_levels, rot_spans, rot_X, rot_agg = run_sweep("rotation")
noi_levels, noi_spans, noi_X, noi_agg = run_sweep("noise")

In [ ]:
show_examples("rotation", rot_levels, rot_spans, rot_X)

In [ ]:
show_examples("noise", noi_levels, noi_spans, noi_X)

## 5. Paper figures -- one per shift

`shift_figure(kind, ...)` produces a **separate figure for noise and for
rotation**, so each can carry more metrics without becoming unreadable.
Rows = models (MAP / SGD point estimates as dotted, Zig-Zag / Boomerang
posteriors as solid). Columns:

1. **accuracy**, one line per digit (+ chance line).
2. **P(true)**, one line per digit. For rotation this is the label-flip
   collapse (a 180-rotated 6 is a 9, so P(true) for 6 falls to ~0); for
   noise it is confidence in the correct class.
3. **accuracy vs mean top-class confidence**, pooled over digits. The
   shaded gap is the calibration story: a point estimate stays confident
   (line pinned near 1) while accuracy falls; a well-calibrated posterior
   bends its confidence down *with* accuracy.
4. **predictive entropy** (nats), pooled, with the uniform-10 reference
   line. A Bayesian model should widen (entropy up) under shift rather
   than staying sharp and wrong.

Per-block `ece` is also stored by `run_sweep` (both per-digit and pooled)
if a scalar-calibration table is wanted.


In [ ]:
RUN_DISPLAY = {"zigzag": "Sticky Zig-Zag", "boomerang": "Sticky Boomerang",
               "map": r"$\beta_{\mathrm{ref}}$", "sgd": "SGD"}

def shift_figure(kind, levels, agg, save=False):
    """One figure per shift. Columns:
      1. accuracy vs level, one line per digit (+ chance line).
      2. P(true) vs level, one line per digit -- for rotation this is the
         label-flip collapse; for noise it is confidence in the correct class.
      3. calibration: accuracy vs mean top-class confidence, POOLED over
         digits, one pair of lines per model. Gap = over/under-confidence.
      4. predictive entropy (nats) vs level, pooled -- does the model widen
         under shift.
    Rows = models (MAP / SGD / Zig-Zag / Boomerang, whichever are loaded).
    """
    labels = list(runs)
    digit_col = {c: plt.cm.tab10(c % 5) for c in CLASSES}
    xlab = SWEEPS[kind]["xlabel"]
    col_titles = ["Accuracy per digit", "P(true digit)",
                  "Accuracy vs confidence", "Predictive entropy"]

    nrows, ncols = len(labels), 4
    fig, axes = plt.subplots(nrows, ncols, figsize=(9.2, 1.85 * nrows + 0.7),
                             sharex="col", squeeze=False)
    fig.subplots_adjust(wspace=0.28, hspace=0.30,
                        left=0.07, right=0.99, top=0.88, bottom=0.16)

    for r, label in enumerate(labels):
        is_point = label in ("map", "sgd")
        ls = ":" if is_point else "-"
        lw = 1.1 if is_point else 1.5

        ax = axes[r][0]
        for c in CLASSES:
            ax.plot(levels, [agg[label][(c, lv)]["acc"] for lv in levels],
                    marker="o", ms=2.6, lw=lw, ls=ls, color=digit_col[c],
                    label=str(c) if r == 0 else None)
        ax.axhline(0.1, color="0.65", lw=0.7, ls="--")
        ax.set_ylim(-0.02, 1.02)

        ax = axes[r][1]
        for c in CLASSES:
            ax.plot(levels, [agg[label][(c, lv)]["p_true"] for lv in levels],
                    marker="o", ms=2.6, lw=lw, ls=ls, color=digit_col[c])
        ax.set_ylim(-0.02, 1.02)

        ax = axes[r][2]
        pa = [agg[label][("pool", lv)]["acc"] for lv in levels]
        pc = [agg[label][("pool", lv)]["conf"] for lv in levels]
        ax.plot(levels, pa, marker="o", ms=3, lw=1.6, color="0.20", label="accuracy")
        ax.plot(levels, pc, marker="s", ms=3, lw=1.6, color="#C44E52", label="confidence")
        ax.fill_between(levels, pa, pc, color="#C44E52", alpha=0.12, lw=0)
        ax.set_ylim(-0.02, 1.02)
        if r == 0:
            ax.legend(fontsize=6.5, loc="lower left", frameon=False, handlelength=1.2)

        ax = axes[r][3]
        ax.plot(levels, [agg[label][("pool", lv)]["entropy"] for lv in levels],
                marker="o", ms=3, lw=1.6, color="#4C72B0")
        ax.axhline(np.log(10), color="0.65", lw=0.7, ls="--")   # uniform-10 entropy
        ax.set_ylim(-0.05, np.log(10) * 1.08)

        for j in range(ncols):
            axes[r][j].grid(alpha=0.25)
            axes[r][j].tick_params(labelsize=6.5, length=2, pad=1)
            if r == 0:
                axes[r][j].set_title(col_titles[j], fontsize=10, pad=4)
            if r == nrows - 1:
                axes[r][j].set_xlabel(xlab, fontsize=10)
            else:
                axes[r][j].tick_params(labelbottom=False)
        axes[r][0].set_ylabel(RUN_DISPLAY.get(label, label), fontsize=10)

    handles = [plt.Line2D([], [], color=digit_col[c], marker="o", ms=3, lw=1.4)
               for c in CLASSES]
    fig.legend(handles, [str(c) for c in CLASSES], loc="lower center",
               title="true digit", fontsize=10, title_fontsize=7.5,
               ncol=len(CLASSES), frameon=False, bbox_to_anchor=(0.5, +0.05),
               handlelength=1.3, columnspacing=1.0)
    fig.suptitle(f"MNIST LeNet5 under {'pixel noise' if kind == 'noise' else 'rotation'}",
                 fontsize=10, y=0.97)

    if save:
        fig.savefig(SAVE_DIR / f"MNIST_{kind}_paper.png", dpi=300, bbox_inches="tight")
        fig.savefig(SAVE_DIR / f"MNIST_{kind}_paper.pdf", bbox_inches="tight")
    plt.show()



In [ ]:
shift_figure("noise", noi_levels, noi_agg, save=True)
shift_figure("rotation", rot_levels, rot_agg, save=True)

## 6. Appendix figure -- full P(digit) bar grid

One figure per shift. Rows = true digit, columns = shift level. Bars =
pooled posterior-mean P(digit), grouped by sampler; whiskers = 10-90% across
the image pool; the true digit is marked green.

In [ ]:
def bar_grid_appendix(kind, levels, agg, save=False):
    DIGITS = np.arange(10)
    labels = [l for l in runs if l not in ("map", "sgd")]   # posterior samplers only
    n = len(labels)
    bw = 0.8 / n
    offs = [(-0.4 + bw / 2) + k * bw for k in range(n)]
    nrows, ncols = len(CLASSES), len(levels)

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(1.55 * ncols, 1.45 * nrows),
                             squeeze=False, sharey=True, sharex=True)
    fig.subplots_adjust(wspace=0.10, hspace=0.16,
                        left=0.07, right=0.99, top=0.92, bottom=0.11)

    for r, c in enumerate(CLASSES):
        for cc, lv in enumerate(levels):
            ax = axes[r][cc]
            ax.axvspan(c - 0.5, c + 0.5, color=TRUE_GREEN, alpha=0.12, zorder=0)
            ax.axvline(c, color=TRUE_GREEN, lw=1.0, zorder=1)
            for k, label in enumerate(labels):
                A = agg[label][(c, lv)]
                x = DIGITS + offs[k]
                edge = ["none"] * 10; ew = [0.0] * 10
                edge[c] = TRUE_GREEN; ew[c] = 1.2
                ax.bar(x, A["mean"], width=bw * (0.92 if n > 1 else 1.0),
                       color=SAMPLER_COLORS.get(label, f"C{k}"),
                       edgecolor=edge, linewidth=ew, zorder=2,
                       label=RUN_DISPLAY.get(label, label) if (r == 0 and cc == 0) else None)
                ax.errorbar(x, A["mean"],
                            yerr=[np.clip(A["mean"] - A["lo"], 0, None),
                                  np.clip(A["hi"] - A["mean"], 0, None)],
                            fmt="none", ecolor="black", elinewidth=0.6, capsize=1.0, zorder=3)
            ax.set_xlim(-0.6, 9.6); ax.set_ylim(0, 1)
            ax.set_xticks(DIGITS); ax.set_yticks([0, 0.5, 1.0])
            ax.tick_params(labelsize=7, length=2, pad=1)
            ax.set_xticklabels(DIGITS if r == nrows - 1 else [], fontsize=10)
            if r == 0:
                ax.set_title(f"{lv:g}", fontsize=12, pad=3)
            if cc == 0:
                ax.set_yticklabels([0, "", 1], fontsize=12)
                ax.set_ylabel(str(c), fontsize=12, rotation=0, labelpad=8, va="center")

    fig.text(0.5, 0.06, "Predicted digit", ha="center", fontsize=12)
    fig.text(0.012, 0.5, "True digit", va="center", rotation="vertical", fontsize=12)
    if n > 1:
        fig.legend(loc="lower center", fontsize=12, ncol=n, frameon=False,
                   bbox_to_anchor=(0.5, +0.0), handlelength=1.4, columnspacing=1.4)

    if save:
        fig.savefig(SAVE_DIR / f"MNIST_{kind}_bars_appendix.png", dpi=300, bbox_inches="tight")
        fig.savefig(SAVE_DIR / f"MNIST_{kind}_bars_appendix.pdf", bbox_inches="tight")
    plt.show()


In [ ]:
bar_grid_appendix("noise", [0.0, 0.3, 0.5, 0.7, 1.0, 1.5], noi_agg, save=True)
bar_grid_appendix("rotation", [0, 15, 30, 45, 90, 120, 180], rot_agg, save=True)